Cell 1 — Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Cell 2 — Imports

In [ ]:
import os
import csv
from pathlib import Path
from PIL import Image, UnidentifiedImageError

Cell 3 — Config

In [ ]:
RAW_DATA_DIR = '/content/drive/MyDrive/new_dataset'
JPG_DATA_DIR = '/content/drive/MyDrive/new_dataset_jpg'
CONVERSION_LOG_PATH = '/content/drive/MyDrive/StyleMind_self/conversion_failures.csv'

CATEGORIES = ['Blazer', 'Dress', 'Formal_Pant', 'Jacket', 'Pants',
              'Shirt', 'Shorts', 'Skirt', 'Top', 'Warmwear']

VALID_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tiff', '.gif'}
JPG_QUALITY = 95


Cell 4 — Conversion function

In [ ]:
def convert_to_jpg():
    """
    Converts every image in RAW_DATA_DIR/<category>/ to .jpg, flattening
    any transparency onto white, and saves into JPG_DATA_DIR/<category>/
    with the same filename stem. Original files are untouched.
    Logs (doesn't crash on) any file that fails to open/convert.
    """
    failures = []
    total_converted = 0

    for category in CATEGORIES:
        src_dir = Path(RAW_DATA_DIR) / category
        dst_dir = Path(JPG_DATA_DIR) / category
        dst_dir.mkdir(parents=True, exist_ok=True)

        if not src_dir.exists():
            print(f"⚠️  Folder not found, skipping: {src_dir}")
            continue

        count = 0
        for fpath in sorted(src_dir.iterdir()):
            if fpath.suffix.lower() not in VALID_EXTENSIONS:
                continue

            out_path = dst_dir / f"{fpath.stem}.jpg"

            try:
                img = Image.open(fpath)

                # Flatten transparency onto white
                if img.mode in ('RGBA', 'LA') or (img.mode == 'P' and 'transparency' in img.info):
                    img = img.convert('RGBA')
                    white_bg = Image.new('RGBA', img.size, (255, 255, 255, 255))
                    img = Image.alpha_composite(white_bg, img).convert('RGB')
                else:
                    img = img.convert('RGB')

                img.save(out_path, 'JPEG', quality=JPG_QUALITY, subsampling=0)
                count += 1
                total_converted += 1

            except (UnidentifiedImageError, OSError) as e:
                failures.append({'filepath': str(fpath), 'reason': str(e)})

        print(f"{category:15}: {count} images converted")

    if failures:
        os.makedirs(os.path.dirname(CONVERSION_LOG_PATH), exist_ok=True)
        with open(CONVERSION_LOG_PATH, 'w', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=['filepath', 'reason'])
            writer.writeheader()
            writer.writerows(failures)
        print(f"\n⚠️  {len(failures)} files failed to convert, logged to {CONVERSION_LOG_PATH}")

    print(f"\n✅ Conversion done: {total_converted} images -> {JPG_DATA_DIR}")
    return total_converted, failures

Cell 5 — To run

In [ ]:
total_converted, failures = convert_to_jpg()

Step 2 — Build Dataset Manifest

Cell 6 — Config for manifest

In [ ]:
JPG_DATA_DIR = '/content/drive/MyDrive/new_dataset_jpg'
MANIFEST_PATH = '/content/drive/MyDrive/StyleMind_self/manifest.csv'
MANIFEST_FAILED_LOG = '/content/drive/MyDrive/StyleMind_self/manifest_failures.csv'

CATEGORIES = ['Blazer', 'Dress', 'Formal_Pant', 'Jacket', 'Pants',
              'Shirt', 'Shorts', 'Skirt', 'Top', 'Warmwear']

In [ ]:
os.makedirs(os.path.dirname(MANIFEST_PATH), exist_ok=True)

Cell 7 — Manifest builder function

In [ ]:
def build_manifest():
    """
    Scans JPG_DATA_DIR/<category>/ and records every valid image as
    (filepath, category). Skips/logs corrupt or unreadable files
    rather than crashing.
    """
    rows = []
    failures = []

    for category in CATEGORIES:
        cat_dir = Path(JPG_DATA_DIR) / category
        if not cat_dir.exists():
            print(f"⚠️  Folder not found, skipping: {cat_dir}")
            continue

        count = 0
        for fpath in sorted(cat_dir.iterdir()):
            if fpath.suffix.lower() != '.jpg':
                continue
            try:
                with Image.open(fpath) as img:
                    img.verify()
                rows.append({'filepath': str(fpath), 'category': category})
                count += 1
            except (UnidentifiedImageError, OSError) as e:
                failures.append({'filepath': str(fpath), 'reason': str(e)})

        print(f"{category:15}: {count} valid images")

    with open(MANIFEST_PATH, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=['filepath', 'category'])
        writer.writeheader()
        writer.writerows(rows)

    if failures:
        with open(MANIFEST_FAILED_LOG, 'w', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=['filepath', 'reason'])
            writer.writeheader()
            writer.writerows(failures)
        print(f"\n⚠️  {len(failures)} corrupt/unreadable files logged to {MANIFEST_FAILED_LOG}")

    print(f"\n✅ Manifest built: {len(rows)} images -> {MANIFEST_PATH}")
    return rows

Cell 8 — To run

In [ ]:
manifest_rows = build_manifest()

Cell 9 — Install ultralytics

In [ ]:
!pip install ultralytics -q

Cell 10 — Reload the manifest (since the session restarted)

In [ ]:
import csv
with open(MANIFEST_PATH) as f:
    manifest_rows = list(csv.DictReader(f))
print(f"Loaded {len(manifest_rows)} rows")

Cell 11 — Config for YOLO checkpoint

In [ ]:
YOLO_CHECKPOINT = '/content/drive/MyDrive/StyleMind_self/deepfashion2_yolov8s-seg.pt'

Cell 11.5 — Download the checkpoint from Hugging Face into StyleMind_self/

In [ ]:
!pip install huggingface_hub -q

In [ ]:
from huggingface_hub import hf_hub_download
import shutil

downloaded_path = hf_hub_download(
    repo_id="Bingsu/adetailer",
    filename="deepfashion2_yolov8s-seg.pt"
)

shutil.copy(downloaded_path, YOLO_CHECKPOINT)
print(f"Checkpoint saved to: {YOLO_CHECKPOINT}")

Cell 12 — Check class list

In [ ]:
from ultralytics import YOLO
model = YOLO(YOLO_CHECKPOINT)
results = model(manifest_rows[0]['filepath'])
print("YOLO-seg class list:", results[0].names)

Cell 13 — Quick multi-category mask test

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

samples_by_category = {}
for row in manifest_rows:
    cat = row['category']
    if cat not in samples_by_category:
        samples_by_category[cat] = row['filepath']
    if len(samples_by_category) == len(CATEGORIES):
        break

fig, axes = plt.subplots(len(samples_by_category), 3, figsize=(10, 3.2 * len(samples_by_category)))

for i, (category, fpath) in enumerate(samples_by_category.items()):
    img = Image.open(fpath).convert('RGB')
    results = model(np.array(img), conf=0.25, verbose=False)
    result = results[0]

    axes[i, 0].imshow(img)
    axes[i, 0].set_title(f'{category} — original')
    axes[i, 0].axis('off')

    if result.masks is None or len(result.masks.data) == 0:
        axes[i, 1].text(0.5, 0.5, 'NO MASK DETECTED', ha='center', va='center', color='red')
        axes[i, 1].axis('off')
        axes[i, 2].text(0.5, 0.5, 'FALLBACK: original', ha='center', va='center', color='red')
        axes[i, 2].axis('off')
        continue

    mask_data = result.masks.data.cpu().numpy()
    areas = [m.sum() for m in mask_data]
    largest_idx = int(np.argmax(areas))
    mask = mask_data[largest_idx]
    mask_resized = np.array(Image.fromarray((mask * 255).astype(np.uint8)).resize(img.size))
    mask_bool = mask_resized > 127

    axes[i, 1].imshow(mask_bool, cmap='gray')
    axes[i, 1].set_title('mask')
    axes[i, 1].axis('off')

    img_arr = np.array(img)
    white_arr = np.full_like(img_arr, 255)
    masked_arr = np.where(mask_bool[..., None], img_arr, white_arr)
    masked_img = Image.fromarray(masked_arr.astype(np.uint8))

    ys, xs = np.where(mask_bool)
    if len(xs) > 0 and len(ys) > 0:
        x0, x1, y0, y1 = xs.min(), xs.max(), ys.min(), ys.max()
        cropped = masked_img.crop((x0, y0, x1 + 1, y1 + 1))
    else:
        cropped = masked_img

    axes[i, 2].imshow(cropped)
    axes[i, 2].set_title('bg removed + cropped')
    axes[i, 2].axis('off')

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/StyleMind_self/mask_test_preview.png', dpi=100, bbox_inches='tight')
plt.show()

Cell 14 — Config for processed output

In [ ]:
PROCESSED_DIR = '/content/drive/MyDrive/processed_dataset'
PROCESSED_MANIFEST_PATH = '/content/drive/MyDrive/StyleMind_self/manifest_processed.csv'
FALLBACK_LOG_PATH = '/content/drive/MyDrive/StyleMind_self/preprocessing_fallbacks.csv'

TARGET_SIZE = 224

Cell 15 — Letterbox resize helper

In [ ]:
def letterbox_resize(img, target_size=TARGET_SIZE, fill_color=(255, 255, 255)):
    w, h = img.size
    scale = target_size / max(w, h)
    new_w, new_h = int(w * scale), int(h * scale)
    img_resized = img.resize((new_w, new_h), Image.LANCZOS)

    canvas = Image.new('RGB', (target_size, target_size), fill_color)
    paste_x = (target_size - new_w) // 2
    paste_y = (target_size - new_h) // 2
    canvas.paste(img_resized, (paste_x, paste_y))
    return canvas

Cell 16 — Full preprocessing function

In [ ]:
import numpy as np

def preprocess_dataset(manifest_rows, yolo_model, conf_threshold=0.25):
    os.makedirs(PROCESSED_DIR, exist_ok=True)
    for category in CATEGORIES:
        os.makedirs(os.path.join(PROCESSED_DIR, category), exist_ok=True)

    fallback_log = []
    processed_manifest = []

    for i, row in enumerate(manifest_rows):
        fpath, category = row['filepath'], row['category']
        out_name = f"{Path(fpath).stem}.jpg"
        out_path = os.path.join(PROCESSED_DIR, category, out_name)

        try:
            img = Image.open(fpath).convert('RGB')

            results = yolo_model(np.array(img), conf=conf_threshold, verbose=False)
            result = results[0]

            if result.masks is None or len(result.masks.data) == 0:
                fallback_log.append({'filepath': fpath, 'reason': 'no mask detected'})
                final_img = img
            else:
                mask_data = result.masks.data.cpu().numpy()
                areas = [m.sum() for m in mask_data]
                largest_idx = int(np.argmax(areas))
                mask = mask_data[largest_idx]

                mask_resized = np.array(
                    Image.fromarray((mask * 255).astype(np.uint8)).resize(img.size)
                )
                mask_bool = mask_resized > 127

                img_arr = np.array(img)
                white_arr = np.full_like(img_arr, 255)
                masked_arr = np.where(mask_bool[..., None], img_arr, white_arr)
                masked_img = Image.fromarray(masked_arr.astype(np.uint8))

                ys, xs = np.where(mask_bool)
                if len(xs) == 0 or len(ys) == 0:
                    fallback_log.append({'filepath': fpath, 'reason': 'empty mask after resize'})
                    final_img = img
                else:
                    x0, x1 = xs.min(), xs.max()
                    y0, y1 = ys.min(), ys.max()
                    final_img = masked_img.crop((x0, y0, x1 + 1, y1 + 1))

            final_img = letterbox_resize(final_img, TARGET_SIZE)
            final_img.save(out_path, 'JPEG', quality=95)
            processed_manifest.append({'filepath': out_path, 'category': category})

        except Exception as e:
            fallback_log.append({'filepath': fpath, 'reason': f'error: {e}'})
            continue

        if (i + 1) % 200 == 0:
            print(f"Processed {i + 1}/{len(manifest_rows)}...")

    with open(PROCESSED_MANIFEST_PATH, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=['filepath', 'category'])
        writer.writeheader()
        writer.writerows(processed_manifest)

    if fallback_log:
        with open(FALLBACK_LOG_PATH, 'w', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=['filepath', 'reason'])
            writer.writeheader()
            writer.writerows(fallback_log)
        print(f"\n⚠️  {len(fallback_log)} images fell back to uncropped (see {FALLBACK_LOG_PATH})")

    print(f"\n✅ Preprocessing done: {len(processed_manifest)} images -> {PROCESSED_DIR}")
    print(f"   Processed manifest saved to {PROCESSED_MANIFEST_PATH}")
    return processed_manifest

Cell 17 — To run

In [ ]:
processed_manifest = preprocess_dataset(manifest_rows, model)

In [ ]:
import pandas as pd
fallbacks = pd.read_csv('/content/drive/MyDrive/StyleMind_self/preprocessing_fallbacks.csv')
print(fallbacks['filepath'].apply(lambda x: x.split('/')[-2]).value_counts())
print(f"\nTotal fallbacks: {len(fallbacks)}")

In [ ]:
import matplotlib.pyplot as plt
import random

fig, axes = plt.subplots(2, 5, figsize=(18, 8))
for ax, category in zip(axes.flat, CATEGORIES):
    cat_dir = Path(PROCESSED_DIR) / category
    sample_file = random.choice(list(cat_dir.iterdir()))
    img = Image.open(sample_file)
    ax.imshow(img)
    ax.set_title(category)
    ax.axis('off')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/StyleMind_self/processed_spotcheck.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
top_fallbacks = fallbacks[fallbacks['filepath'].str.contains('/Top/')]
print(top_fallbacks.to_string())

In [ ]:
import matplotlib.pyplot as plt

problem_files = [
    '/content/drive/MyDrive/new_dataset_jpg/Top/top011.jpg',
    '/content/drive/MyDrive/new_dataset_jpg/Top/top081.jpg',
    '/content/drive/MyDrive/new_dataset_jpg/Top/top102.jpg',
    '/content/drive/MyDrive/new_dataset_jpg/Top/top108.jpg',
    '/content/drive/MyDrive/new_dataset_jpg/Top/top135.jpg',
    '/content/drive/MyDrive/new_dataset_jpg/Top/top163.jpg',
]

fig, axes = plt.subplots(1, 6, figsize=(18, 4))
for ax, fpath in zip(axes, problem_files):
    img = Image.open(fpath)
    ax.imshow(img)
    ax.set_title(Path(fpath).name, fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/StyleMind_self/top_fallback_samples.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

check_categories = ['Formal_Pant', 'Shorts', 'Skirt']

fig, axes = plt.subplots(len(check_categories), 2, figsize=(8, 4 * len(check_categories)))

for i, category in enumerate(check_categories):
    # Grab the same filename from both raw (JPG-converted) and processed folders
    raw_dir = Path(JPG_DATA_DIR) / category
    proc_dir = Path(PROCESSED_DIR) / category

    sample_name = sorted(raw_dir.iterdir())[0].name  # first file, e.g. formal_pant001.jpg
    raw_path = raw_dir / sample_name
    proc_path = proc_dir / sample_name

    raw_img = Image.open(raw_path)
    proc_img = Image.open(proc_path)

    axes[i, 0].imshow(raw_img)
    axes[i, 0].set_title(f'{category} — ORIGINAL ({raw_img.size[0]}x{raw_img.size[1]})')
    axes[i, 0].axis('off')

    axes[i, 1].imshow(proc_img)
    axes[i, 1].set_title(f'{category} — PROCESSED ({proc_img.size[0]}x{proc_img.size[1]})')
    axes[i, 1].axis('off')

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/StyleMind_self/blur_check.png', dpi=100, bbox_inches='tight')
plt.show()

Step 6 — CLIP Zero-Shot Labeling

Cell 18 — Install open_clip

In [ ]:
!pip install open_clip_torch -q

Cell 19 — Load CLIP model

In [ ]:
import open_clip
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    'ViT-B-32', pretrained='laion2b_s34b_b79k'
)
clip_model = clip_model.to(device).eval()
clip_tokenizer = open_clip.get_tokenizer('ViT-B-32')

print(f"CLIP loaded on {device}")

Cell 20 — Config: classes, prompts, rules

In [ ]:
TEXTURE_CLASSES = ['solid', 'striped', 'floral', 'graphic', 'embroidered', 'pleated', 'checkered']
SEASON_CLASSES = ['summer', 'winter', 'fall', 'all-season']

# Fuller descriptive prompts — CLIP zero-shot accuracy is sensitive to phrasing,
# richer prompts generally beat single words
TEXTURE_PROMPTS = {
    'solid': 'a photo of a solid-colored garment with no pattern',
    'striped': 'a photo of a striped garment',
    'floral': 'a photo of a garment with a floral pattern',
    'graphic': 'a photo of a garment with a graphic print design',
    'embroidered': 'a photo of a garment with embroidered detailing',
    'pleated': 'a photo of a pleated garment with folded fabric texture',
    'checkered': 'a photo of a checkered or plaid patterned garment',
}
SEASON_PROMPTS = {
    'summer': 'a photo of a lightweight summer clothing item',
    'winter': 'a photo of a heavy warm winter clothing item',
    'fall': 'a photo of a medium-weight fall or autumn clothing item',

}

# Season rule mapping — categories here skip CLIP entirely
SEASON_RULES = {
    'Warmwear': 'winter',
    'Jacket': 'winter',
    'Shorts': 'summer',
}

# Confidence fallback threshold for season CLIP predictions
SEASON_CONFIDENCE_MARGIN = 0.03  # top-1 vs top-2 similarity gap; below this -> 'all-season'

Cell 21 — Precompute text embeddings

In [ ]:
def get_text_embeddings(prompts_dict):
    labels = list(prompts_dict.keys())
    texts = list(prompts_dict.values())
    tokens = clip_tokenizer(texts).to(device)
    with torch.no_grad():
        embeddings = clip_model.encode_text(tokens)
        embeddings = embeddings / embeddings.norm(dim=-1, keepdim=True)
    return labels, embeddings

texture_labels, texture_embeddings = get_text_embeddings(TEXTURE_PROMPTS)
season_labels, season_embeddings = get_text_embeddings(SEASON_PROMPTS)

Cell 22 — Per-image labeling function

In [ ]:
def label_image(img_path, category):
    img = clip_preprocess(Image.open(img_path).convert('RGB')).unsqueeze(0).to(device)
    with torch.no_grad():
        img_embedding = clip_model.encode_image(img)
        img_embedding = img_embedding / img_embedding.norm(dim=-1, keepdim=True)

    # --- Texture: always CLIP-based ---
    tex_sims = (img_embedding @ texture_embeddings.T).squeeze(0)
    tex_sims_sorted, tex_idx_sorted = tex_sims.sort(descending=True)
    texture_label = texture_labels[tex_idx_sorted[0].item()]
    texture_confidence = tex_sims_sorted[0].item()

    # --- Season: rule-based override, else CLIP with confidence fallback ---
    if category in SEASON_RULES:
        season_label = SEASON_RULES[category]
        season_confidence = 1.0  # rule-based, fully confident by definition
        season_source = 'rule'
    else:
        season_sims = (img_embedding @ season_embeddings.T).squeeze(0)
        season_sims_sorted, season_idx_sorted = season_sims.sort(descending=True)
        top1_score = season_sims_sorted[0].item()
        top2_score = season_sims_sorted[1].item()

        if (top1_score - top2_score) < SEASON_CONFIDENCE_MARGIN:
            season_label = 'all-season'
            season_source = 'low_confidence_fallback'
        else:
            season_label = season_labels[season_idx_sorted[0].item()]
            season_source = 'clip'
        season_confidence = top1_score

    return {
        'texture': texture_label,
        'texture_confidence': round(texture_confidence, 4),
        'season': season_label,
        'season_confidence': round(season_confidence, 4),
        'season_source': season_source,
    }

Cell 23 — Run labeling on the full processed dataset

In [ ]:
CLIP_LABELED_MANIFEST_PATH = '/content/drive/MyDrive/StyleMind_self/manifest_labeled.csv'

with open(PROCESSED_MANIFEST_PATH) as f:
    processed_rows = list(csv.DictReader(f))

labeled_rows = []
for i, row in enumerate(processed_rows):
    labels = label_image(row['filepath'], row['category'])
    labeled_rows.append({**row, **labels})

    if (i + 1) % 200 == 0:
        print(f"Labeled {i + 1}/{len(processed_rows)}...")

with open(CLIP_LABELED_MANIFEST_PATH, 'w', newline='') as f:
    fieldnames = ['filepath', 'category', 'texture', 'texture_confidence',
                  'season', 'season_confidence', 'season_source']
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(labeled_rows)

print(f"\n✅ Labeling done: {len(labeled_rows)} images -> {CLIP_LABELED_MANIFEST_PATH}")

**Step 7 — Manual QA on the CLIP labels.**

Cell 24 — Label distribution + confidence overview

In [ ]:
labeled_df = pd.read_csv(CLIP_LABELED_MANIFEST_PATH)

print("=== TEXTURE distribution ===")
print(labeled_df['texture'].value_counts())
print(f"\nTexture confidence — mean: {labeled_df['texture_confidence'].mean():.4f}, "
      f"min: {labeled_df['texture_confidence'].min():.4f}, "
      f"max: {labeled_df['texture_confidence'].max():.4f}")

print("\n=== SEASON distribution ===")
print(labeled_df['season'].value_counts())
print("\n=== SEASON source breakdown ===")
print(labeled_df['season_source'].value_counts())

Cell 26 — Check the actual confidence-gap distribution for season

In [ ]:
clip_season_rows = labeled_df[labeled_df['season_source'] != 'rule']
print(clip_season_rows['season_confidence'].describe())

import matplotlib.pyplot as plt
plt.hist(clip_season_rows['season_confidence'], bins=30)
plt.xlabel('Top-1 season similarity score')
plt.ylabel('Count')
plt.axvline(x=clip_season_rows['season_confidence'].median(), color='red', linestyle='--', label='median')
plt.legend()
plt.savefig('/content/drive/MyDrive/StyleMind_self/season_confidence_dist.png', dpi=100, bbox_inches='tight')
plt.show()

Cell 22 (revised) — updated season logic + gap tracking

In [ ]:
SEASON_CONFIDENCE_MARGIN = 0.008  # recalibrated based on observed score distribution (std ≈ 0.02)

def label_image(img_path, category):
    img = clip_preprocess(Image.open(img_path).convert('RGB')).unsqueeze(0).to(device)
    with torch.no_grad():
        img_embedding = clip_model.encode_image(img)
        img_embedding = img_embedding / img_embedding.norm(dim=-1, keepdim=True)

    tex_sims = (img_embedding @ texture_embeddings.T).squeeze(0)
    tex_sims_sorted, tex_idx_sorted = tex_sims.sort(descending=True)
    texture_label = texture_labels[tex_idx_sorted[0].item()]
    texture_confidence = tex_sims_sorted[0].item()

    if category in SEASON_RULES:
        season_label = SEASON_RULES[category]
        season_confidence = 1.0
        season_gap = None
        season_source = 'rule'
    else:
        season_sims = (img_embedding @ season_embeddings.T).squeeze(0)
        season_sims_sorted, season_idx_sorted = season_sims.sort(descending=True)
        top1_score = season_sims_sorted[0].item()
        top2_score = season_sims_sorted[1].item()
        gap = top1_score - top2_score

        if gap < SEASON_CONFIDENCE_MARGIN:
            season_label = 'all-season'
            season_source = 'low_confidence_fallback'
        else:
            season_label = season_labels[season_idx_sorted[0].item()]
            season_source = 'clip'
        season_confidence = top1_score
        season_gap = round(gap, 4)

    return {
        'texture': texture_label,
        'texture_confidence': round(texture_confidence, 4),
        'season': season_label,
        'season_confidence': round(season_confidence, 4),
        'season_gap': season_gap,
        'season_source': season_source,
    }

Cell 23 (rerun with updated fieldnames)

In [ ]:
with open(PROCESSED_MANIFEST_PATH) as f:
    processed_rows = list(csv.DictReader(f))

labeled_rows = []
for i, row in enumerate(processed_rows):
    labels = label_image(row['filepath'], row['category'])
    labeled_rows.append({**row, **labels})
    if (i + 1) % 200 == 0:
        print(f"Labeled {i + 1}/{len(processed_rows)}...")

with open(CLIP_LABELED_MANIFEST_PATH, 'w', newline='') as f:
    fieldnames = ['filepath', 'category', 'texture', 'texture_confidence',
                  'season', 'season_confidence', 'season_gap', 'season_source']
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(labeled_rows)

print(f"\n✅ Re-labeling done: {len(labeled_rows)} images -> {CLIP_LABELED_MANIFEST_PATH}")

Cell 24b — check where "fall" ranks even when it doesn't win

In [ ]:
# For clip-sourced rows, how often was 'fall' the runner-up (top-2) even if it never won?
clip_rows = labeled_df[labeled_df['season_source'] == 'clip']
print(clip_rows['season'].value_counts())

# Also worth checking: what's the actual season_labels list and their prompts?
print(season_labels)

Cell 24c — sanity check season prompts on a handful of known-obvious images

In [ ]:
import random

sample_df = labeled_df[labeled_df['category'].isin(['Blazer', 'Dress', 'Shirt', 'Top', 'Skirt'])].groupby('category', group_keys=False).apply(
    lambda x: x.sample(min(4, len(x)), random_state=42)
).reset_index(drop=True)

season_check_labels = ['summer', 'winter', 'fall']
season_check_prompts = [SEASON_PROMPTS[s] for s in season_check_labels]
season_check_tokens = clip_tokenizer(season_check_prompts).to(device)
with torch.no_grad():
    season_check_embeddings = clip_model.encode_text(season_check_tokens)
    season_check_embeddings = season_check_embeddings / season_check_embeddings.norm(dim=-1, keepdim=True)

for _, row in sample_df.iterrows():
    img = clip_preprocess(Image.open(row['filepath']).convert('RGB')).unsqueeze(0).to(device)
    with torch.no_grad():
        img_embedding = clip_model.encode_image(img)
        img_embedding = img_embedding / img_embedding.norm(dim=-1, keepdim=True)
    sims = (img_embedding @ season_check_embeddings.T).squeeze(0).tolist()
    scores = dict(zip(season_check_labels, [round(s, 4) for s in sims]))
    print(f"{row['category']:12s} | {scores}  (file: {row['filepath'].split('/')[-1]})")

In [ ]:
sample_df = labeled_df[labeled_df['category'].isin(['Pants', 'Formal_Pant'])].groupby('category', group_keys=False).apply(
    lambda x: x.sample(min(6, len(x)), random_state=42)
).reset_index(drop=True)

for _, row in sample_df.iterrows():
    img = clip_preprocess(Image.open(row['filepath']).convert('RGB')).unsqueeze(0).to(device)
    with torch.no_grad():
        img_embedding = clip_model.encode_image(img)
        img_embedding = img_embedding / img_embedding.norm(dim=-1, keepdim=True)
    sims = (img_embedding @ season_check_embeddings.T).squeeze(0).tolist()
    scores = dict(zip(season_check_labels, [round(s, 4) for s in sims]))
    print(f"{row['category']:12s} | {scores}  (file: {row['filepath'].split('/')[-1]})")

Cell 22 (final) — season prompts without all-season

In [ ]:
TEXTURE_CLASSES = ['solid', 'striped', 'floral', 'graphic', 'embroidered', 'pleated', 'checkered']
SEASON_CLASSES = ['summer', 'winter', 'fall']  # all-season removed — now purely a fallback outcome

TEXTURE_PROMPTS = {
    'solid': 'a photo of a solid-colored garment with no pattern',
    'striped': 'a photo of a striped garment',
    'floral': 'a photo of a garment with a floral pattern',
    'graphic': 'a photo of a garment with a graphic print design',
    'embroidered': 'a photo of a garment with embroidered detailing',
    'pleated': 'a photo of a pleated garment with folded fabric texture',
    'checkered': 'a photo of a checkered or plaid patterned garment',
}
SEASON_PROMPTS = {
    'summer': 'a photo of a lightweight summer clothing item',
    'winter': 'a photo of a heavy warm winter clothing item',
    'fall': 'a photo of a medium-weight fall or autumn clothing item',
}

SEASON_RULES = {
    'Warmwear': 'winter',
    'Jacket': 'winter',
    'Shorts': 'summer',
}

SEASON_CONFIDENCE_MARGIN = 0.008  # keep the recalibrated margin from before

texture_labels = list(TEXTURE_PROMPTS.keys())
texture_tokens = clip_tokenizer([TEXTURE_PROMPTS[t] for t in texture_labels]).to(device)
with torch.no_grad():
    texture_embeddings = clip_model.encode_text(texture_tokens)
    texture_embeddings = texture_embeddings / texture_embeddings.norm(dim=-1, keepdim=True)

season_labels = list(SEASON_PROMPTS.keys())  # now just ['summer', 'winter', 'fall']
season_tokens = clip_tokenizer([SEASON_PROMPTS[s] for s in season_labels]).to(device)
with torch.no_grad():
    season_embeddings = clip_model.encode_text(season_tokens)
    season_embeddings = season_embeddings / season_embeddings.norm(dim=-1, keepdim=True)

def label_image(img_path, category):
    img = clip_preprocess(Image.open(img_path).convert('RGB')).unsqueeze(0).to(device)
    with torch.no_grad():
        img_embedding = clip_model.encode_image(img)
        img_embedding = img_embedding / img_embedding.norm(dim=-1, keepdim=True)

    tex_sims = (img_embedding @ texture_embeddings.T).squeeze(0)
    tex_sims_sorted, tex_idx_sorted = tex_sims.sort(descending=True)
    texture_label = texture_labels[tex_idx_sorted[0].item()]
    texture_confidence = tex_sims_sorted[0].item()

    if category in SEASON_RULES:
        season_label = SEASON_RULES[category]
        season_confidence = 1.0
        season_gap = None
        season_source = 'rule'
    else:
        season_sims = (img_embedding @ season_embeddings.T).squeeze(0)
        season_sims_sorted, season_idx_sorted = season_sims.sort(descending=True)
        top1_score = season_sims_sorted[0].item()
        top2_score = season_sims_sorted[1].item()
        gap = top1_score - top2_score

        if gap < SEASON_CONFIDENCE_MARGIN:
            season_label = 'all-season'
            season_source = 'low_confidence_fallback'
        else:
            season_label = season_labels[season_idx_sorted[0].item()]
            season_source = 'clip'
        season_confidence = top1_score
        season_gap = round(gap, 4)

    return {
        'texture': texture_label,
        'texture_confidence': round(texture_confidence, 4),
        'season': season_label,
        'season_confidence': round(season_confidence, 4),
        'season_gap': season_gap,
        'season_source': season_source,
    }

Cell 23 — rerun full labeling

In [ ]:
with open(PROCESSED_MANIFEST_PATH) as f:
    processed_rows = list(csv.DictReader(f))

labeled_rows = []
for i, row in enumerate(processed_rows):
    labels = label_image(row['filepath'], row['category'])
    labeled_rows.append({**row, **labels})
    if (i + 1) % 200 == 0:
        print(f"Labeled {i + 1}/{len(processed_rows)}...")

with open(CLIP_LABELED_MANIFEST_PATH, 'w', newline='') as f:
    fieldnames = ['filepath', 'category', 'texture', 'texture_confidence',
                  'season', 'season_confidence', 'season_gap', 'season_source']
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(labeled_rows)

print(f"\n✅ Re-labeling done: {len(labeled_rows)} images -> {CLIP_LABELED_MANIFEST_PATH}")

Cell 25 — visual spot-check grid, sampled per category

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import math

def spot_check_grid(df, n_per_category=4, categories=None, seed=42):
    if categories is None:
        categories = df['category'].unique()

    sample = df[df['category'].isin(categories)].groupby('category', group_keys=False).apply(
        lambda x: x.sample(min(n_per_category, len(x)), random_state=seed)
    ).reset_index(drop=True)

    n = len(sample)
    cols = 4
    rows = math.ceil(n / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3.2, rows * 3.6))
    axes = axes.flatten()

    for i, (_, row) in enumerate(sample.iterrows()):
        img = mpimg.imread(row['filepath'])
        axes[i].imshow(img)
        axes[i].axis('off')
        title = f"{row['category']}\ntex: {row['texture']} ({row['texture_confidence']:.2f})\nseason: {row['season']} ({row['season_source']})"
        axes[i].set_title(title, fontsize=8)

    for j in range(n, len(axes)):
        axes[j].axis('off')

    plt.tight_layout()
    plt.savefig('/content/drive/MyDrive/StyleMind_self/spot_check_grid.png', dpi=100, bbox_inches='tight')
    plt.show()

# Round 1: everything, few per category
spot_check_grid(labeled_df, n_per_category=3)

Cell 27 — stratified split

In [ ]:
from sklearn.model_selection import train_test_split

labeled_df = pd.read_csv(CLIP_LABELED_MANIFEST_PATH)

# 70/15/15 split, stratified by category
train_df, temp_df = train_test_split(
    labeled_df, test_size=0.30, stratify=labeled_df['category'], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df['category'], random_state=42
)

print(f"Train: {len(train_df)} ({len(train_df)/len(labeled_df):.1%})")
print(f"Val:   {len(val_df)} ({len(val_df)/len(labeled_df):.1%})")
print(f"Test:  {len(test_df)} ({len(test_df)/len(labeled_df):.1%})")

# Sanity check: category proportions should be near-identical across splits
print("\n=== Category % by split ===")
for name, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    print(f"\n{name}:")
    print((df['category'].value_counts(normalize=True) * 100).round(1))

# Save splits
train_df.to_csv('/content/drive/MyDrive/StyleMind_self/manifest_train.csv', index=False)
val_df.to_csv('/content/drive/MyDrive/StyleMind_self/manifest_val.csv', index=False)
test_df.to_csv('/content/drive/MyDrive/StyleMind_self/manifest_test.csv', index=False)

print("\n✅ Splits saved.")

Cell 28 — check texture/season didn't end up lopsided as a side effect

In [ ]:
print("=== Texture % by split ===")
for name, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    print(f"\n{name}:")
    print((df['texture'].value_counts(normalize=True) * 100).round(1))

print("\n=== Season % by split ===")
for name, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    print(f"\n{name}:")
    print((df['season'].value_counts(normalize=True) * 100).round(1))

Cell 29 — encode labels for all three heads

In [ ]:
import tensorflow as tf
import numpy as np

IMG_SIZE = 224
BATCH_SIZE = 32

# Fit label encodings on the FULL labeled_df so class indices are consistent
# across train/val/test (important — don't fit separately per split)
category_classes = sorted(labeled_df['category'].unique())
texture_classes = sorted(labeled_df['texture'].unique())
season_classes = sorted(labeled_df['season'].unique())

category_to_idx = {c: i for i, c in enumerate(category_classes)}
texture_to_idx = {t: i for i, t in enumerate(texture_classes)}
season_to_idx = {s: i for i, s in enumerate(season_classes)}

print(f"Category classes ({len(category_classes)}): {category_classes}")
print(f"Texture classes ({len(texture_classes)}): {texture_classes}")
print(f"Season classes ({len(season_classes)}): {season_classes}")

# Save these mappings — you'll need them at inference time to decode predictions
import json
label_maps = {
    'category': category_to_idx,
    'texture': texture_to_idx,
    'season': season_to_idx,
}
with open('/content/drive/MyDrive/StyleMind_self/label_maps.json', 'w') as f:
    json.dump(label_maps, f, indent=2)
print("\n✅ Label maps saved.")

Cell 30 — tf.data.Dataset pipeline with augmentation:

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def load_and_preprocess(filepath, category_idx, texture_idx, season_idx):
    img = tf.io.read_file(filepath)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32) / 255.0
    return img, {
        'category': category_idx,
        'texture': texture_idx,
        'season': season_idx,
    }

# Augmentation layer — train only
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.08),
    tf.keras.layers.RandomZoom(0.1),
])

def augment(img, labels):
    return data_augmentation(img, training=True), labels

def make_dataset(df, training=False):
    filepaths = df['filepath'].values
    category_idx = df['category'].map(category_to_idx).values
    texture_idx = df['texture'].map(texture_to_idx).values
    season_idx = df['season'].map(season_to_idx).values

    ds = tf.data.Dataset.from_tensor_slices(
        (filepaths, category_idx, texture_idx, season_idx)
    )
    ds = ds.map(load_and_preprocess, num_parallel_calls=AUTOTUNE)

    if training:
        ds = ds.shuffle(buffer_size=len(df), reshuffle_each_iteration=True)
        ds = ds.map(augment, num_parallel_calls=AUTOTUNE)

    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(AUTOTUNE)
    return ds

train_ds = make_dataset(train_df, training=True)
val_ds = make_dataset(val_df, training=False)
test_ds = make_dataset(test_df, training=False)

print("✅ Datasets built.")
for imgs, labels in train_ds.take(1):
    print("Image batch shape:", imgs.shape)
    print("Category label batch shape:", labels['category'].shape)
    print("Texture label batch shape:", labels['texture'].shape)
    print("Season label batch shape:", labels['season'].shape)

**Step 10 — Model architecture.**

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, Model

NUM_CATEGORY = len(category_classes)   # 10
NUM_TEXTURE = len(texture_classes)     # 7
NUM_SEASON = len(season_classes)       # 4

base_model = MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet',
    pooling='avg'
)
base_model.trainable = False  # freeze for phase 1

inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base_model(inputs, training=False)
x = layers.Dropout(0.3)(x)

# Shared trunk
shared = layers.Dense(256, activation='relu')(x)
shared = layers.Dropout(0.3)(shared)

# Category head
cat_branch = layers.Dense(128, activation='relu')(shared)
category_output = layers.Dense(NUM_CATEGORY, activation='softmax', name='category')(cat_branch)

# Texture head
tex_branch = layers.Dense(128, activation='relu')(shared)
texture_output = layers.Dense(NUM_TEXTURE, activation='softmax', name='texture')(tex_branch)

# Season head
sea_branch = layers.Dense(64, activation='relu')(shared)
season_output = layers.Dense(NUM_SEASON, activation='softmax', name='season')(sea_branch)

model = Model(inputs=inputs, outputs=[category_output, texture_output, season_output])
model.summary()

Cell 32 — compile:

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss={
        'category': 'sparse_categorical_crossentropy',
        'texture': 'sparse_categorical_crossentropy',
        'season': 'sparse_categorical_crossentropy',
    },
    loss_weights={'category': 1.0, 'texture': 1.0, 'season': 1.0},
    metrics={
        'category': 'accuracy',
        'texture': 'accuracy',
        'season': 'accuracy',
    }
)

Step 11 — Model Training (Phase 1: Frozen Base)

Cell 33 — Train with Early Stopping & Checkpointing

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

CHECKPOINT_PATH = '/content/drive/MyDrive/StyleMind_self/best_model_phase1.keras'

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    filepath=CHECKPOINT_PATH,
    monitor='val_loss',
    save_best_only=True
)

EPOCHS = 30  # early stopping will likely cut this short

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[early_stop, checkpoint]
)

Cell 34 — Unfreeze and recompile

In [ ]:
# Load best phase-1 weights first, in case this is a fresh runtime
model.load_weights('/content/drive/MyDrive/StyleMind_self/best_model_phase1.keras')

# Unfreeze the base model
base_model.trainable = True

# Freeze all layers EXCEPT the last N — keeps early generic feature extractors
# (edges, textures, colors) intact, only lets high-level layers adapt
FINE_TUNE_AT = 100  # MobileNetV2 has ~155 layers total; adjust if needed

for layer in base_model.layers[:FINE_TUNE_AT]:
    layer.trainable = False

# Recompile with a much lower learning rate — critical, otherwise
# fine-tuning destroys the pretrained weights instead of gently adapting them
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss={
        'category': 'sparse_categorical_crossentropy',
        'texture': 'sparse_categorical_crossentropy',
        'season': 'sparse_categorical_crossentropy',
    },
    loss_weights={'category': 1.0, 'texture': 1.0, 'season': 1.0},
    metrics={
        'category': 'accuracy',
        'texture': 'accuracy',
        'season': 'accuracy',
    }
)

# Confirm how many params are now trainable
model.summary()

Cell 35 — Fine-tune training run


In [ ]:
CHECKPOINT_PATH_PHASE2 = '/content/drive/MyDrive/StyleMind_self/best_model_phase2.keras'

early_stop_ft = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

checkpoint_ft = ModelCheckpoint(
    filepath=CHECKPOINT_PATH_PHASE2,
    monitor='val_loss',
    save_best_only=True
)

FINE_TUNE_EPOCHS = 15  # fine-tuning typically needs fewer epochs than phase 1

history_finetune = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=FINE_TUNE_EPOCHS,
    callbacks=[early_stop_ft, checkpoint_ft]
)

Cell 36 — Load best model and evaluate on test set

In [ ]:
# Reload phase 1's best weights explicitly, to be certain we're evaluating
# the right checkpoint (not whatever's currently in memory after phase 2 attempts)
model.load_weights('/content/drive/MyDrive/StyleMind_self/best_model_phase1.keras')

test_results = model.evaluate(test_ds, return_dict=True)

print("=== Test Set Results ===")
for key, value in test_results.items():
    print(f"{key}: {value:.4f}")

Step 14 — Per-Class Analysis (Confusion Matrices & Precision/Recall)

Cell 37 — Generate predictions on the test set

In [ ]:
import numpy as np

# Get predictions for all three heads on the test set
y_pred = model.predict(test_ds)
# y_pred is a list: [category_preds, texture_preds, season_preds]
# each shape (num_test_samples, num_classes) — softmax probabilities

category_pred_idx = np.argmax(y_pred[0], axis=1)
texture_pred_idx = np.argmax(y_pred[1], axis=1)
season_pred_idx = np.argmax(y_pred[2], axis=1)

# Get true labels — pulled directly from test_df to guarantee correct order
# (test_ds was built with training=False, so order is deterministic/unshuffled)
category_true_idx = test_df['category'].map(category_to_idx).values
texture_true_idx = test_df['texture'].map(texture_to_idx).values
season_true_idx = test_df['season'].map(season_to_idx).values

print(f"Predictions generated for {len(category_pred_idx)} test images.")

Cell 38 — Classification reports (precision/recall/F1 per class)

In [ ]:
from sklearn.metrics import classification_report

print("=" * 60)
print("CATEGORY — per-class report")
print("=" * 60)
print(classification_report(
    category_true_idx, category_pred_idx,
    target_names=category_classes, digits=3
))

print("=" * 60)
print("TEXTURE — per-class report")
print("=" * 60)
print(classification_report(
    texture_true_idx, texture_pred_idx,
    target_names=texture_classes, digits=3
))

print("=" * 60)
print("SEASON — per-class report")
print("=" * 60)
print(classification_report(
    season_true_idx, season_pred_idx,
    target_names=season_classes, digits=3
))

Cell 39 — Confusion matrices (visual)

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

def plot_confusion(y_true, y_pred, class_names, title, filename):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(max(6, len(class_names)*0.9), max(5, len(class_names)*0.8)))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title(title)
    plt.tight_layout()
    plt.savefig(f'/content/drive/MyDrive/StyleMind_self/{filename}', dpi=100, bbox_inches='tight')
    plt.show()

plot_confusion(category_true_idx, category_pred_idx, category_classes,
                'Category Confusion Matrix', 'confusion_category.png')
plot_confusion(texture_true_idx, texture_pred_idx, texture_classes,
                'Texture Confusion Matrix', 'confusion_texture.png')
plot_confusion(season_true_idx, season_pred_idx, season_classes,
                'Season Confusion Matrix', 'confusion_season.png')

1. Compute class weights per head (inverse frequency, computed from the training set only):

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

def get_class_weights(train_df, col, class_to_idx):
    labels = train_df[col].map(class_to_idx).values
    classes = np.unique(labels)
    weights = compute_class_weight(class_weight='balanced', classes=classes, y=labels)
    return dict(zip(classes, weights))

texture_class_weights = get_class_weights(train_df, 'texture', texture_to_idx)
season_class_weights = get_class_weights(train_df, 'season', season_to_idx)

print("Texture weights:", {texture_classes[k]: round(v, 2) for k, v in texture_class_weights.items()})
print("Season weights:", {season_classes[k]: round(v, 2) for k, v in season_class_weights.items()})

2. Apply weights via a custom weighted loss.

In [ ]:
def weighted_sparse_ce(class_weights_dict, num_classes):
    weight_vector = tf.constant(
        [class_weights_dict.get(i, 1.0) for i in range(num_classes)], dtype=tf.float32
    )
    def loss_fn(y_true, y_pred):
        y_true = tf.cast(y_true, tf.int32)
        sample_weights = tf.gather(weight_vector, y_true)
        base_loss = tf.keras.losses.sparse_categorical_crossentropy(y_true, y_pred)
        return base_loss * sample_weights
    return loss_fn

texture_loss = weighted_sparse_ce(texture_class_weights, len(texture_classes))
season_loss = weighted_sparse_ce(season_class_weights, len(season_classes))

3. Recompile from best_model_phase1.keras

In [ ]:
model.load_weights('/content/drive/MyDrive/StyleMind_self/best_model_phase1.keras')

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss={
        'category': 'sparse_categorical_crossentropy',  # leave category unweighted, it's fine
        'texture': texture_loss,
        'season': season_loss,
    },
    metrics={'category': 'accuracy', 'texture': 'accuracy', 'season': 'accuracy'}
)

history_weighted = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True, monitor='val_loss'),
        tf.keras.callbacks.ModelCheckpoint(
            '/content/drive/MyDrive/StyleMind_self/best_model_weighted.keras',
            save_best_only=True, monitor='val_loss'
        ),
    ]
)

In [ ]:
model.load_weights('/content/drive/MyDrive/StyleMind_self/best_model_phase1.keras')

# Explicitly re-freeze — critical, since base_model.trainable was left True
# from the earlier fine-tuning attempt and load_weights doesn't reset it
base_model.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss={
        'category': 'sparse_categorical_crossentropy',
        'texture': texture_loss,
        'season': season_loss,
    },
    metrics={'category': 'accuracy', 'texture': 'accuracy', 'season': 'accuracy'}
)

early_stop_w = tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True, monitor='val_loss')
checkpoint_w = tf.keras.callbacks.ModelCheckpoint(
    '/content/drive/MyDrive/StyleMind_self/best_model_weighted.keras',
    save_best_only=True, monitor='val_loss'
)

history_weighted = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=[early_stop_w, checkpoint_w]
)

In [ ]:
texture_class_weights_soft = {k: v**0.5 for k, v in texture_class_weights.items()}
# season_class_weights can stay as-is — it worked fine

In [ ]:
model.load_weights('/content/drive/MyDrive/StyleMind_self/best_model_phase1.keras')
base_model.trainable = False  # don't skip this again

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss={
        'category': 'sparse_categorical_crossentropy',
        'texture': weighted_sparse_ce(texture_class_weights_soft, len(texture_classes)),
        'season': season_loss,  # unchanged, already worked
    },
    metrics={'category': 'accuracy', 'texture': 'accuracy', 'season': 'accuracy'}
)

history_soft = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True, monitor='val_loss'),
        tf.keras.callbacks.ModelCheckpoint(
            '/content/drive/MyDrive/StyleMind_self/best_model_texture_soft.keras',
            save_best_only=True, monitor='val_loss'
        ),
    ]
)

In [ ]:
model.load_weights('/content/drive/MyDrive/StyleMind_self/best_model_texture_soft.keras')
test_results = model.evaluate(test_ds, return_dict=True)

Cell 1 — Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Cell 2 — Load model + label maps

In [ ]:
import json
import numpy as np
import tensorflow as tf
from PIL import Image
import matplotlib.pyplot as plt

DRIVE_BASE = '/content/drive/MyDrive/StyleMind_self'
MODEL_PATH = f'{DRIVE_BASE}/best_model_phase1.keras'
LABEL_MAP_PATH = f'{DRIVE_BASE}/label_maps.json'
IMG_SIZE = 224

print("Loading model...")
model = tf.keras.models.load_model(MODEL_PATH, compile=False)

with open(LABEL_MAP_PATH) as f:
    label_maps = json.load(f)

idx_to_category = {v: k for k, v in label_maps['category'].items()}
idx_to_texture = {v: k for k, v in label_maps['texture'].items()}
idx_to_season = {v: k for k, v in label_maps['season'].items()}

print(f"Loaded. {len(idx_to_category)} categories, "
      f"{len(idx_to_texture)} textures, {len(idx_to_season)} seasons.")

Cell 3 — predict() and display_result()

In [ ]:
def predict(image_path):
    img = Image.open(image_path).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
    img_array = np.expand_dims(np.array(img, dtype=np.float32), axis=0)
    img_array = img_array / 255.0   # match training normalization (Cell 84: tf.cast(img, tf.float32) / 255.0)

    category_pred, texture_pred, season_pred = model.predict(img_array, verbose=0)

    category_idx = int(np.argmax(category_pred[0]))
    texture_idx = int(np.argmax(texture_pred[0]))
    season_idx = int(np.argmax(season_pred[0]))

    return {
        'category': idx_to_category[category_idx],
        'category_confidence': round(float(category_pred[0][category_idx]), 4),
        'texture': idx_to_texture[texture_idx],
        'texture_confidence': round(float(texture_pred[0][texture_idx]), 4),
        'season': idx_to_season[season_idx],
        'season_confidence': round(float(season_pred[0][season_idx]), 4),
    }


def display_result(image_path, result):
    img = Image.open(image_path).convert('RGB')
    fig, (ax_img, ax_info) = plt.subplots(1, 2, figsize=(9, 5),
                                           gridspec_kw={'width_ratios': [1, 1]})
    ax_img.imshow(img)
    ax_img.axis('off')
    ax_img.set_title('Input', fontsize=12)

    ax_info.axis('off')
    lines = [
        f"Category:  {result['category']}  ({result['category_confidence']:.0%})",
        f"Texture:   {result['texture']}  ({result['texture_confidence']:.0%})",
        f"Season:    {result['season']}  ({result['season_confidence']:.0%})",
    ]
    ax_info.text(0.0, 0.6, "\n\n".join(lines), fontsize=13,
                 verticalalignment='center', family='monospace')
    plt.tight_layout()
    plt.show()

Cell 4 — Run the interactive demo

In [ ]:
from google.colab import files

print("Upload a clothing image (click the stop ■ button to end).")
while True:
    uploaded = files.upload()
    if not uploaded:
        break
    for filename in uploaded.keys():
        result = predict(filename)
        print(f"\n--- {filename} ---")
        display_result(filename, result)

In [ ]:
import pandas as pd

test_df = pd.read_csv('/content/drive/MyDrive/StyleMind_self/manifest_test.csv')
sample = test_df.sample(5, random_state=1)

for _, row in sample.iterrows():
    result = predict(row['filepath'])
    print(f"\n{row['filepath'].split('/')[-1]}")
    print(f"  TRUE:  category={row['category']}, texture={row['texture']}, season={row['season']}")
    print(f"  PRED:  category={result['category']}, texture={result['texture']}, season={result['season']}")

In [ ]:
for _, row in sample.iterrows():
    result = predict(row['filepath'])
    print(f"\n{row['filepath'].split('/')[-1]}")
    print(f"  TRUE:  category={row['category']}, texture={row['texture']}, season={row['season']}")
    print(f"  PRED:  category={result['category']}, texture={result['texture']}, season={result['season']}")


Cell 0 — Rebuild test_ds and label mappings from saved files


In [ ]:
import pandas as pd
import json
import tensorflow as tf

DRIVE_BASE = '/content/drive/MyDrive/StyleMind_self'
IMG_SIZE = 224
BATCH_SIZE = 32

# Load label maps (saved during training) and test manifest
with open(f'{DRIVE_BASE}/label_maps.json') as f:
    label_maps = json.load(f)

category_to_idx = label_maps['category']
texture_to_idx = label_maps['texture']
season_to_idx = label_maps['season']

category_classes = sorted(category_to_idx, key=category_to_idx.get)
texture_classes = sorted(texture_to_idx, key=texture_to_idx.get)
season_classes = sorted(season_to_idx, key=season_to_idx.get)

test_df = pd.read_csv(f'{DRIVE_BASE}/manifest_test.csv')

# Rebuild test_ds exactly as it was built during training (no augmentation, no shuffle)
AUTOTUNE = tf.data.AUTOTUNE

def load_and_preprocess(filepath, category_idx, texture_idx, season_idx):
    img = tf.io.read_file(filepath)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32) / 255.0
    return img, {
        'category': category_idx,
        'texture': texture_idx,
        'season': season_idx,
    }

def make_dataset(df, training=False):
    filepaths = df['filepath'].values
    cat_idx = df['category'].map(category_to_idx).values
    tex_idx = df['texture'].map(texture_to_idx).values
    sea_idx = df['season'].map(season_to_idx).values

    ds = tf.data.Dataset.from_tensor_slices((filepaths, cat_idx, tex_idx, sea_idx))
    ds = ds.map(load_and_preprocess, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(AUTOTUNE)
    return ds

test_ds = make_dataset(test_df, training=False)

print(f"✅ test_ds rebuilt: {len(test_df)} images, "
      f"{len(category_classes)} categories, {len(texture_classes)} textures, {len(season_classes)} seasons.")

Cell A — Reload phase 1 and generate predictions on the test set


In [ ]:
import numpy as np

# Explicitly reload phase 1 weights to be certain we're evaluating the right checkpoint
model.load_weights('/content/drive/MyDrive/StyleMind_self/best_model_phase1.keras')

y_pred = model.predict(test_ds)
# y_pred is a list: [category_preds, texture_preds, season_preds]

category_pred_idx = np.argmax(y_pred[0], axis=1)
texture_pred_idx = np.argmax(y_pred[1], axis=1)
season_pred_idx = np.argmax(y_pred[2], axis=1)

category_true_idx = test_df['category'].map(category_to_idx).values
texture_true_idx = test_df['texture'].map(texture_to_idx).values
season_true_idx = test_df['season'].map(season_to_idx).values

print(f"Predictions generated for {len(category_pred_idx)} test images.")

Cell B — Classification reports (precision/recall/F1 per class)


In [ ]:
from sklearn.metrics import classification_report

print("=" * 60)
print("CATEGORY — per-class report (phase 1)")
print("=" * 60)
print(classification_report(
    category_true_idx, category_pred_idx,
    target_names=category_classes, digits=3
))

print("=" * 60)
print("TEXTURE — per-class report (phase 1)")
print("=" * 60)
print(classification_report(
    texture_true_idx, texture_pred_idx,
    target_names=texture_classes, digits=3
))

print("=" * 60)
print("SEASON — per-class report (phase 1)")
print("=" * 60)
print(classification_report(
    season_true_idx, season_pred_idx,
    target_names=season_classes, digits=3
))

Cell C — Confusion matrices (visual, saved for the results doc)


In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

def plot_confusion(y_true, y_pred, class_names, title, filename):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(max(6, len(class_names)*0.9), max(5, len(class_names)*0.8)))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title(title)
    plt.tight_layout()
    plt.savefig(f'/content/drive/MyDrive/StyleMind_self/{filename}', dpi=100, bbox_inches='tight')
    plt.show()

plot_confusion(category_true_idx, category_pred_idx, category_classes,
                'Category Confusion Matrix (Phase 1 - Final)', 'confusion_category_final.png')
plot_confusion(texture_true_idx, texture_pred_idx, texture_classes,
                'Texture Confusion Matrix (Phase 1 - Final)', 'confusion_texture_final.png')
plot_confusion(season_true_idx, season_pred_idx, season_classes,
                'Season Confusion Matrix (Phase 1 - Final)', 'confusion_season_final.png')